In [1]:
# ── IMPORTS (run this first) ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

In [2]:
# ── Q4 + Q5 – Load dataset and display first 5 rows ──────────
df = pd.read_csv('posa_shaktoi.csv')
print('Columns:', df.columns.tolist())
df.head()

Columns: ['num_preg', 'glucose_conc', 'diastolic_bp', 'thickness', 'insulin', 'bmi', 'diab_pred', 'age', 'diabetes']


,num_preg,glucose_conc,diastolic_bp,thickness,insulin,bmi,diab_pred,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# ── Q1 – Class balance of target variable ─────────────────────
print('=== Class Counts ===')
print(df.iloc[:, -1].value_counts())
print('\n=== Class Balance (%) ===')
print(df.iloc[:, -1].value_counts(normalize=True) * 100)

=== Class Counts ===
diabetes
0    500
1    268
Name: count, dtype: int64

=== Class Balance (%) ===
diabetes
0    65.104167
1    34.895833
Name: proportion, dtype: float64


In [4]:
# ── Q2 – Feature with highest correlation with diabetes ───────
corr = df.corr().iloc[:, -1].drop(df.columns[-1]).sort_values(ascending=False)
print('=== Correlation with Target ===')
print(corr)
print(f'\nHighest: {corr.idxmax()} ({corr.max():.4f})')

=== Correlation with Target ===
glucose_conc    0.466581
bmi             0.292695
age             0.238356
num_preg        0.221898
diab_pred       0.173844
insulin         0.130548
thickness       0.074752
diastolic_bp    0.065068
Name: diabetes, dtype: float64

Highest: glucose_conc (0.4666)


In [5]:
# ── Q3 – Reason for imputing: zeros are biologically impossible
zero_cols = ['glucose_conc', 'diastolic_bp', 'thickness', 'insulin', 'bmi']
print('=== Zero counts in medical columns (these are missing values) ===')
print((df[zero_cols] == 0).sum())

=== Zero counts in medical columns (these are missing values) ===
glucose_conc      5
diastolic_bp     35
thickness       227
insulin         374
bmi              11
dtype: int64


In [6]:
# ── Q6 – Replace 0 with NaN in biologically invalid columns ───
df[zero_cols] = df[zero_cols].replace(0, np.nan)
print('=== Null counts AFTER replacing zeros with NaN ===')
print(df.isnull().sum())

=== Null counts AFTER replacing zeros with NaN ===
num_preg          0
glucose_conc      5
diastolic_bp     35
thickness       227
insulin         374
bmi              11
diab_pred         0
age               0
diabetes          0
dtype: int64


In [7]:
# ── Q7 + Q9 – Median imputation + null check after ───────────
# Median used because columns are right-skewed (especially insulin).
# Median is robust to outliers unlike mean.
df[zero_cols] = df[zero_cols].fillna(df[zero_cols].median())

print('=== Null counts AFTER imputation (should all be 0) ===')
print(df.isnull().sum())

=== Null counts AFTER imputation (should all be 0) ===
num_preg        0
glucose_conc    0
diastolic_bp    0
thickness       0
insulin         0
bmi             0
diab_pred       0
age             0
diabetes        0
dtype: int64


In [8]:
# ── Q8 – Train/Test Split + Scaling ──────────────────────────
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train shape: {X_train_scaled.shape}')
print(f'Test shape:  {X_test_scaled.shape}')

Train shape: (614, 8)
Test shape:  (154, 8)


In [9]:
# ── Q10 + Q11 – Logistic Regression + Evaluation ─────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr  = lr.predict_proba(X_test_scaled)[:, 1]

print('=' * 45)
print('       LOGISTIC REGRESSION RESULTS')
print('=' * 45)
print(f'Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_lr):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred_lr):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob_lr):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr))

       LOGISTIC REGRESSION RESULTS
Accuracy  : 0.7078
Precision : 0.6000
Recall    : 0.5000
F1 Score  : 0.5455
ROC-AUC   : 0.8130

Confusion Matrix:
[[82 18]
 [27 27]]


In [10]:
# ── Q12 + Q13 + Q14 – Random Forest with Hyperparameter Tuning
param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print('Best Hyperparameters:', grid_search.best_params_)
best_rf = grid_search.best_estimator_

Best Hyperparameters: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}


In [11]:
# ── Q15 – Random Forest Evaluation ────────────────────────────
y_pred_rf = best_rf.predict(X_test_scaled)
y_prob_rf  = best_rf.predict_proba(X_test_scaled)[:, 1]

print('=' * 45)
print('       RANDOM FOREST RESULTS')
print('=' * 45)
print(f'Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_rf):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_prob_rf):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_rf))

       RANDOM FOREST RESULTS
Accuracy  : 0.7403
Precision : 0.6667
Recall    : 0.5185
F1 Score  : 0.5833
ROC-AUC   : 0.8144

Confusion Matrix:
[[86 14]
 [26 28]]


In [ ]:
# ── Q16 + Q18 – Feature Importance + Plot ─────────────────────
feature_names = X.columns.tolist()
importances   = best_rf.feature_importances_

feat_df = pd.DataFrame({
    'Feature':    feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print('=== Top 3 Most Important Features ===')
print(feat_df.head(3).to_string(index=False))

plt.figure(figsize=(8, 5))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Random Forest - Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

In [12]:
# ── Q17 – Clinical relevance of top 3 features ────────────────
print("""
1. glucose_conc  - High blood glucose is the defining marker of diabetes.
                   Elevated levels directly indicate insulin resistance.

2. bmi           - Obesity is the strongest lifestyle risk factor for
                   Type 2 diabetes. Excess fat reduces insulin sensitivity.

3. age           - Risk increases with age as insulin secretion declines
                   and physical activity typically decreases.
""")


1. glucose_conc  - High blood glucose is the defining marker of diabetes.
                   Elevated levels directly indicate insulin resistance.

2. bmi           - Obesity is the strongest lifestyle risk factor for
                   Type 2 diabetes. Excess fat reduces insulin sensitivity.

3. age           - Risk increases with age as insulin secretion declines
                   and physical activity typically decreases.



In [ ]:
# ── Q19 – Final test accuracy after retraining best model ─────
best_rf.fit(X_train_scaled, y_train)
final_preds = best_rf.predict(X_test_scaled)
final_acc   = accuracy_score(y_test, final_preds)
print(f'Final Test Accuracy: {final_acc:.4f}')

In [13]:
# ── Q20 – One improvement with more time ──────────────────────
print("""
Improvement: Apply SMOTE (Synthetic Minority Oversampling Technique)
to handle class imbalance (~34% diabetic vs ~66% non-diabetic).
The model sees fewer diabetic samples during training, hurting recall
for the positive class. Balancing classes before training would improve
the model's ability to correctly identify diabetic patients - which is
more clinically critical than overall accuracy.
""")


Improvement: Apply SMOTE (Synthetic Minority Oversampling Technique)
to handle class imbalance (~34% diabetic vs ~66% non-diabetic).
The model sees fewer diabetic samples during training, hurting recall
for the positive class. Balancing classes before training would improve
the model's ability to correctly identify diabetic patients - which is
more clinically critical than overall accuracy.

